# MINI Cells — Experiment 005B: Recurrent Optimization Factorial Ablation

This experiment runs the full 2^3 factorial design over three MiniTextNCA training factors: RMSNorm, GRU carry bias +2, and auxiliary stage losses [0.1, 0.2]. Each cell consumes exactly 500,000 training tokens using the same TinyStories corpus identity, model seed and training schedule as Experiment 005.

When two T4 GPUs are available, two independent cells run concurrently — one process per GPU.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path('/kaggle/working/mini-cells')
os.chdir('/kaggle/working')

if not (ROOT / '.git').exists():
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ArcheLabs/mini-cells.git', str(ROOT),
    ], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=ROOT, check=True)
    subprocess.run(['git', 'switch', 'main'], cwd=ROOT, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=ROOT, check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
subprocess.run([sys.executable, '-c', "import torch; print('cuda=', torch.cuda.is_available()); print('count=', torch.cuda.device_count()); print('gpus=', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"], check=True)


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/research/01-foundations/test_language_bridge.py', 'tests/research/01-foundations/test_language_ablation.py', '-q'
], cwd=ROOT, check=True)


In [ ]:
subprocess.run([sys.executable, 'scripts/research/run_consumer_language_ablation.py'], cwd=ROOT, check=True)


In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

OUT = ROOT / 'results' / 'consumer-language-ablation-v1'
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
display(Markdown(f"## {decision['status']}: {decision['diagnosis']}"))
display(pd.read_csv(OUT / 'factorial-results.csv').sort_values('validation_ppl'))
display(pd.read_csv(OUT / 'factorial-effects.csv').sort_values('abs_effect_nll', ascending=False))
for name in [
    'factorial-ppl.png',
    'factorial-learning-curves.png',
    'main-effects.png',
    'interaction-effects.png',
    'triple-interaction.png',
    'replication.png',
]:
    display(Image(filename=str(OUT / name)))
display(Markdown((OUT / 'generation-progression.md').read_text(encoding='utf-8')))


In [ ]:
# Set this to True only after reviewing decision.json and the factorial plots.
PUBLISH = False
if PUBLISH:
    subprocess.run([
        sys.executable, 'scripts/research/publish_experiment_005b_results.py', '--push'
    ], cwd=ROOT, check=True)
